# Install

In [1]:
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 40.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0


In [13]:
import os

if not os.path.exists('Sketch-to-Image-by-Pix2Pix'):
    !git clone -b dev https://github.com/lqb464/Sketch-to-Image-by-Pix2Pix

from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

os.chdir('Sketch-to-Image-by-Pix2Pix/')
!pip install -q -r requirements.txt

Cloning into 'Sketch-to-Image-by-Pix2Pix'...
remote: Enumerating objects: 2816, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 2816 (delta 107), reused 77 (delta 37), pack-reused 2643 (from 2)
Receiving objects: 100% (2816/2816), 8.50 MiB | 20.92 MiB/s, done.
Resolving deltas: 100% (1764/1764), done.


In [3]:
!pip install -q numpy==2.0.0
!pip install -q "torch==2.4.0" "torchvision==0.19.0" --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 865.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 54.0 MB/s eta 0:00:0000:01:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.2 MB/s eta 

In [4]:
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("WANDB_API_KEY")
print(len(key))
print(key)

86
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


# Training


In [19]:
import yaml, random, numpy as np, torch

CONFIG_PHASE1 = "./configs/model6_resnet_pretrained.yaml"
CONFIG_PHASE2 = "./configs/model6_resnet_finetune.yaml"

def load_config(path):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    SEED = cfg.get("seed", 42)
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    return cfg

In [6]:
def build_train_cmd(cfg):
    freeze_flag     = "--freeze_encoder" if cfg.get("freeze_encoder", True)  else ""
    no_dropout_flag = "--no_dropout"      if cfg.get("no_dropout", False)     else ""
    continue_flag   = "--continue_train"  if cfg.get("continue_train", False) else ""
    epoch_count     = f'--epoch_count {cfg["epoch_count"]}' if cfg.get("epoch_count") else ""
    flags = " ".join(f for f in [freeze_flag, no_dropout_flag, continue_flag, 
                                  epoch_count, "--no_html"] if f)
    return (
        f'python train.py'
        f' --dataroot         /kaggle/input/sketch2image-dataset/cufs'
        f' --name             {cfg["name"]}'
        f' --model            {cfg["model"]}'
        f' --direction        {cfg["direction"]}'
        f' --netG             {cfg["netG"]}'
        f' --backbone         {cfg["backbone"]}'
        f' --netD             {cfg["netD"]}'
        f' --norm             {cfg["norm"]}'
        f' --gan_mode         {cfg["gan_mode"]}'
        f' --lambda_L1        {cfg["lambda_L1"]}'
        f' --batch_size       {cfg["batch_size"]}'
        f' --lr               {cfg["lr"]}'
        f' --lr_policy        {cfg.get("lr_policy", "linear")}'
        f' --beta1            {cfg.get("beta1", 0.5)}'
        f' --n_epochs         {cfg.get("n_epochs")}'
        f' --n_epochs_decay   {cfg.get("n_epochs_decay")}'
        f' --num_threads      {cfg["num_threads"]}'
        f' --load_size        {cfg["load_size"]}'
        f' --crop_size        {cfg["crop_size"]}'
        f' --print_freq       {cfg["print_freq"]}'
        f' --display_freq     {cfg["display_freq"]}'
        f' --save_latest_freq {cfg["save_latest_freq"]}'
        f' --save_epoch_freq  {cfg["save_epoch_freq"]}'
        f' --dataset_mode     {cfg.get("dataset_mode", "aligned")}'
        f' --wandb_project_name {cfg["wandb_project_name"]}'
        f' --use_wandb'
        f' {flags}'
    )

In [7]:
with open("util/visualizer.py") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if 'wandb.init(' in line and 'self.wandb_run' in line:
        indent = ' ' * 16
        new_lines.append(indent + 'wandb.login(key=os.environ.get("WANDB_API_KEY"), relogin=True)\n')
    new_lines.append(line)

with open("util/visualizer.py", "w") as f:
    f.writelines(new_lines)
print("Done")

Done


In [20]:
# ── Giai đoạn 1: freeze encoder ──────────────────────────────
cfg1 = load_config(CONFIG_PHASE1)
cfg1['gan_mode'] = 'lsgan'
cfg1['name'] = 'm6_resnet18_lambda100_basic_lsgan'
print(f"=== GIAI ĐOẠN 1: {cfg1['name']} ===")
print(f"freeze_encoder : {cfg1.get('freeze_encoder')}")
print(f"Epochs         : {cfg1['n_epochs']} + {cfg1['n_epochs_decay']}")
print(f"lr             : {cfg1['lr']}")

train_cmd1 = build_train_cmd(cfg1)
!{train_cmd1}

=== GIAI ĐOẠN 1: m6_resnet18_lambda100_basic_lsgan ===
freeze_encoder : True
Epochs         : 50 + 0
lr             : 0.0002
----------------- Options ---------------
                 backbone: resnet18                      
               batch_size: 4                             	[default: 1]
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: /kaggle/input/sketch2image-dataset/cufs	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 1                             
          eval_batch_size: 1             

In [22]:
# ── Giai đoạn 2: unfreeze encoder ────────────────────────────
cfg2 = load_config(CONFIG_PHASE2)
cfg2['gan_mode'] = 'lsgan'
cfg2['name'] = 'm6_resnet18_lambda100_basic_lsgan'
print(f"\n=== GIAI ĐOẠN 2: {cfg2['name']} ===")
print(f"freeze_encoder : {cfg2.get('freeze_encoder')}")
print(f"Epochs         : {cfg2['n_epochs']} + {cfg2['n_epochs_decay']}")
print(f"lr             : {cfg2['lr']}")

train_cmd2 = build_train_cmd(cfg2)
!{train_cmd2}


=== GIAI ĐOẠN 2: m6_resnet18_lambda100_basic_lsgan ===
freeze_encoder : False
Epochs         : 100 + 50
lr             : 5e-05
----------------- Options ---------------
                 backbone: resnet18                      
               batch_size: 4                             	[default: 1]
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: True                          	[default: False]
                crop_size: 256                           
                 dataroot: /kaggle/input/sketch2image-dataset/cufs	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 51                            	[default: 1]
        